Shows how one can generate text given a prompt and some hyperparameters, using either minGPT or huggingface/transformers

In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from mingpt.model import GPT
from mingpt.utils import set_seed
from mingpt.bpe import BPETokenizer
set_seed(3407)

/root/miniconda3/envs/vila1.5/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
use_mingpt = True # use minGPT or huggingface/transformers model?
model_type = 'gpt2-large'
device = 'cuda'

In [3]:
if use_mingpt:
    model = GPT.from_pretrained(model_type)
else:
    model = GPT2LMHeadModel.from_pretrained(model_type)
    model.config.pad_token_id = model.config.eos_token_id # suppress a warning

# ship model to device and set to eval mode
model.to(device)
model.eval()

number of parameters: 1482.35M
number of parameters: 774.03M
hf params: 653, local params: 653


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0): Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=1280, out_features=3840, bias=True)
          (c_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
          (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          (act): NewGELU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (1): MoEBlock(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
   

In [4]:

def generate(prompt='', num_samples=10, steps=20, do_sample=True):
        
    # tokenize the input prompt into integer input sequence
    if use_mingpt:
        tokenizer = BPETokenizer()
        if prompt == '':
            # to create unconditional samples...
            # manually create a tensor with only the special <|endoftext|> token
            # similar to what openai's code does here https://github.com/openai/gpt-2/blob/master/src/generate_unconditional_samples.py
            x = torch.tensor([[tokenizer.encoder.encoder['<|endoftext|>']]], dtype=torch.long)
        else:
            x = tokenizer(prompt).to(device)
    else:
        tokenizer = GPT2Tokenizer.from_pretrained(model_type)
        if prompt == '': 
            # to create unconditional samples...
            # huggingface/transformers tokenizer special cases these strings
            prompt = '<|endoftext|>'
        encoded_input = tokenizer(prompt, return_tensors='pt').to(device)
        x = encoded_input['input_ids']
    
    # we'll process all desired num_samples in a batch, so expand out the batch dim
    x = x.expand(num_samples, -1)

    # forward the model `steps` times to get samples, in a batch
    y = model.generate(x, max_new_tokens=steps, do_sample=do_sample, top_k=40)
    
    for i in range(num_samples):
        out = tokenizer.decode(y[i].cpu().squeeze())
        print('-'*80)
        print(out)
        

In [5]:
generate(prompt='Andrej Karpathy, the', num_samples=10, steps=20)

--------------------------------------------------------------------------------
Andrej Karpathy, the CEO of MMM Capital said that we need to know that there is a need.

"
--------------------------------------------------------------------------------
Andrej Karpathy, the leader of the opposition Parti de Gauche, in a radio interview on Sunday said he would be
--------------------------------------------------------------------------------
Andrej Karpathy, the author of the post, says the most recent version of the map, made last month, includes the
--------------------------------------------------------------------------------
Andrej Karpathy, the head of the Institute for Social Research of the Polish Academy of Sciences in Warsaw, has been using this
--------------------------------------------------------------------------------
Andrej Karpathy, the director of the Center for Eastern European Studies at U of T and a co-author of the study
-----------------------------------------

In [7]:
import numpy as np

In [ ]:
expert_trigger = model.get_expert_trigger_count()
expert_trigger / np.sum(expert_trigger, axis=-1)[:, None]

array([[0.27484848, 0.27772727, 0.26530303, 0.18212121],
       [0.25136364, 0.15121212, 0.26318182, 0.33424242],
       [0.24121212, 0.26863636, 0.27242424, 0.21772727],
       [0.22090909, 0.25757576, 0.29136364, 0.23015152],
       [0.23515152, 0.25893939, 0.27242424, 0.23348485],
       [0.31454545, 0.2430303 , 0.16287879, 0.27954545],
       [0.17545455, 0.26015152, 0.29924242, 0.26515152],
       [0.24348485, 0.16363636, 0.29      , 0.30287879],
       [0.21560606, 0.26590909, 0.24393939, 0.27454545],
       [0.23424242, 0.28060606, 0.21984848, 0.26530303],
       [0.24878788, 0.24621212, 0.29227273, 0.21272727],
       [0.24106061, 0.20166667, 0.25287879, 0.30439394],
       [0.21954545, 0.22318182, 0.21075758, 0.34651515],
       [0.23378788, 0.29909091, 0.2830303 , 0.18409091],
       [0.15439394, 0.26636364, 0.28606061, 0.29318182],
       [0.29575758, 0.26848485, 0.23560606, 0.20015152],
       [0.24863636, 0.19651515, 0.22909091, 0.32575758],
       [0.275     , 0.20818182,